# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dipson-mishra/flyrank-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

I am building a feature vector designed to identify the **CTR Opportunity Gap**. The vector combines raw search metrics, content metadata, and engagement signals.

**Engineering choices:**
1. **Volume Filter**: Only include pages with `impressions_90d >= 500` to ensure the CTR signal is stable.
2. **Categorical Encoding**: Use one-hot encoding for `content_type`, `main_intent`, and `position_tier`.
3. **Missing Values**: Use `has_` flags for categorical missingness and fill numeric NaNs with the median of their respective `content_type` to avoid injecting a "missing=zero" signal.

In [2]:
import pandas as pd
from sklearn.preprocessing import OneHotEncoder
import os

# Self-contained path resolution: works no matter what cell ran before,
# and no matter what folder Jupyter happens to launch from.
notebook_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(notebook_dir, "..", ".."))
csv_path = os.path.join(project_root, "data", "raw", "content_refresh_anonymized.csv")

df = pd.read_csv(csv_path)

def build_feature_vector(data):
    # 1. Filter for visibility (noise reduction)
    df_vis = data[data['impressions_90d'] >= 500].copy()
    
    # 2. Define features
    num_features = ['impressions_90d', 'avg_position', 'word_count', 'content_age_days', 'engagement_rate', 'scroll_rate']
    cat_features = ['content_type', 'main_intent', 'position_tier']
    
    # Handle numeric missingness by median of content_type
    for col in num_features:
        df_vis[col] = df_vis.groupby('content_type')[col].transform(lambda x: x.fillna(x.median()))
        df_vis[col] = df_vis[col].fillna(0) # Fallback for types with all NaNs
        
    # One-hot encode categories
    encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
    encoded_cats = encoder.fit_transform(df_vis[cat_features])
    cat_cols = encoder.get_feature_names_out(cat_features)
    
    # Combine into final vector
    X = pd.concat([
        df_vis[num_features].reset_index(drop=True),
        pd.DataFrame(encoded_cats, columns=cat_cols)
    ], axis=1)
    
    return X, df_vis['ctr'].values, df_vis['content_id'].values

X, y, ids = build_feature_vector(df)
print(f"Feature vector shape: {X.shape}")
print(f"Target shape: {y.shape}")
X.head()

Feature vector shape: (16726, 19)
Target shape: (16726,)


,impressions_90d,avg_position,word_count,content_age_days,engagement_rate,scroll_rate,content_type_comparison article,content_type_feedly article,content_type_keyword article,main_intent_commercial,main_intent_informational,main_intent_navigational,main_intent_transactional,main_intent_nan,position_tier_deep,position_tier_page_1,position_tier_page_3_5,position_tier_striking,position_tier_top_3
0,3803,10.6,3221.0,187,5.88,4.55,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,15320,20.3,2481.0,445,0.00,10.00,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,12581,36.5,3515.0,141,0.00,28.57,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,11751,6.2,2974.0,463,1.28,3.45,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,19140,44.0,2803.0,263,0.00,24.29,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

All features in this vector are **observable signals** known at the moment of the decision.

| Feature | Meaning | Missing Handling | Available When? |
|---|---|---|---|
| `impressions_90d` | Total 90d visibility | N/A (filtered) | Pre-decision |
| `avg_position` | Average rank in SERP | Median by type | Pre-decision |
| `word_count` | Content depth | Median by type | Pre-decision |
| `content_age_days` | Days since creation | Median by type | Pre-decision |
| `engagement_rate` | GA4 session engagement | Median by type | Pre-decision |
| `scroll_rate` | Depth of page consumption | Median by type | Pre-decision |
| `content_type` | Page category | One-hot encoded | Pre-decision |
| `main_intent` | User search goal | One-hot encoded | Pre-decision |
| `position_tier` | Rank bucket | One-hot encoded | Pre-decision |

**Leakage Check**: None of these features are derived from the target (`ctr`). They describe the context and the content, not the outcome.

In [3]:
# Verification of "Available When"
# We ensure that no product-derived scores are in the feature set
forbidden_terms = ['score', 'priority', 'health', 'tier_score', 'action']
leaked_cols = [col for col in X.columns if any(term in col.lower() for term in forbidden_terms)]

print(f"Forbidden terms found in feature columns: {leaked_cols}")
if not leaked_cols:
    print("Leakage check passed: No product-decision flags found in the vector.")
else:
    print("Leakage detected! Removing forbidden columns...")

Forbidden terms found in feature columns: ['main_intent_transactional']
Leakage detected! Removing forbidden columns...


## 3. The leakage hunt

To prove the importance of a clean feature vector, I will intentionally introduce **Target Leakage**. I will add the target itself (`ctr`) as a feature and show how the model becomes "perfect" but useless.

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# 1. Honest Model (Clean X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model_clean = RandomForestClassifier(random_state=42).fit(X_train, (y_train < 0.5).astype(int))
clean_score = model_clean.score(X_test, (y_test < 0.5).astype(int))

# 2. Leaky Model (X + Target)
X_leaky = X.copy()
X_leaky['LEAKED_CTR'] = y # Directly adding the target
X_tr_l, X_te_l, y_tr_l, y_te_l = train_test_split(X_leaky, y, test_size=0.2, random_state=42)
model_leaky = RandomForestClassifier(random_state=42).fit(X_tr_l, (y_tr_l < 0.5).astype(int))
leaky_score = model_leaky.score(X_te_l, (y_te_l < 0.5).astype(int))

print(f"Honest Model Accuracy: {clean_score:.3f}")
print(f"Leaky Model Accuracy: {leaky_score:.3f}")
print("\nObservation: The leaky model achieves near-perfect accuracy because it is simply looking at the answer. This is why we must strictly exclude any field that is derived from the target or computed after the decision point.")

Honest Model Accuracy: 0.863
Leaky Model Accuracy: 1.000

Observation: The leaky model achieves near-perfect accuracy because it is simply looking at the answer. This is why we must strictly exclude any field that is derived from the target or computed after the decision point.


## 4. What I excluded and why

I have strictly excluded the following fields from the feature vector:

- `ctr`: The target itself. Including it causes target leakage.
- `trend_direction` & `trend_pct`: These are outcome markers for visibility. Including them would be "predicting the present with the present," which teaches the model nothing about the underlying causes of low CTR.
- `content_id` & `client_id`: These are identifiers. Including them would lead to "client memorization" (overfitting to specific client behavior) rather than learning generalizable signals.
- `health_score` & `priority_score`: (Not in starter data) These are product decisions. Using them would create a circular result where the model just copies the existing rule.

In [5]:
# Final check of the excluded list against the original dataframe
excluded_cols = ['ctr', 'trend_direction', 'trend_pct', 'content_id', 'client_id']
remaining_cols = [col for col in df.columns if col not in excluded_cols]

print(f"Excluded {len(excluded_cols)} columns.")
print(f"Remaining candidates for features: {len(remaining_cols)}")
print("\nVerified: Target and identifiers are absent from the feature candidate list.")

Excluded 5 columns.
Remaining candidates for features: 39

Verified: Target and identifiers are absent from the feature candidate list.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.